In [1]:
import json
import os

import geopandas as gpd
import sqlalchemy

from lyra_plugins.metrics.accessibility_jobs import metric

In [2]:
engine = sqlalchemy.create_engine(
    f"postgresql+psycopg2://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}@{os.environ['POSTGRES_HOST']}:{os.environ['POSTGRES_PORT']}/{os.environ['POSTGRES_DB']}"
)

In [3]:
with engine.connect() as conn:
    df = gpd.read_postgis(
        """
        SELECT census_2020_ageb.cvegeo, census_2020_ageb.geometry FROM census_2020_ageb
        INNER JOIN census_2020_mun
            ON census_2020_ageb.cve_mun = census_2020_mun.cvegeo
        WHERE census_2020_mun.cve_met = '02.2.03'
        """,
        conn,
        geom_col="geometry",
    ).set_index("cvegeo")

In [4]:
j = json.loads(df.to_json())

In [5]:
metric(
    j, patterns=[r"^\d{6}$"], year=2025, edge_weights="travel_time", max_weight=20 * 60
)

TypeError: metric() missing 1 required keyword-only argument: 'context'